In [60]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver


In [61]:
llm = ChatOllama(model='qwen2.5:0.5b-instruct',temperature=0.2,max_token=100)

In [62]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explaination: str
    


In [63]:
def generate_explaination(state:JokeState):
    prompt = f'give the 2 line explaination for the following joke - {state['joke']}'
    res = llm.invoke(prompt)
    return {'explaination':res.content}

def generate_joke(state:JokeState):
    prompt = f'generate a small joke on the topic - {state['topic']}'
    res = llm.invoke(prompt)
    return {'joke':res.content}
     


In [64]:
checkpointer = InMemorySaver()
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

workflow = graph.compile(checkpointer=checkpointer)


In [65]:
thread_id = '1'
config1 = {'configurable':{'thread_id':thread_id}}
while True:
    user_message = input("Ask topic for joke : ")
    print("Human : ",user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    res = workflow.invoke({'topic':user_message},config=config1)

    print('AI : ',res)

Human :  class
AI :  {'topic': 'class', 'joke': "Why did the class break up?\n\nBecause they couldn't agree on what subject to study!", 'explaination': "The punchline is that the class broke up because they couldn't agree on what subject to study, which is a common reason why students often have arguments or conflicts in their classes."}
Human :  exit


In [66]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'class', 'joke': "Why did the class break up?\n\nBecause they couldn't agree on what subject to study!", 'explaination': "The punchline is that the class broke up because they couldn't agree on what subject to study, which is a common reason why students often have arguments or conflicts in their classes."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03f2-3aab-6bdc-8002-098a54a9c340'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-14T13:21:05.301588+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03f2-3152-6103-8001-3df8323a44e9'}}, tasks=(), interrupts=())

In [73]:
list(workflow.get_state_history(config1)) 

[StateSnapshot(values={'topic': 'class', 'joke': "Why did the student go to the teacher's office? Because he wanted to complain about his boring lecture!", 'explaination': 'The joke plays on the idea that the student is complaining about the boring lecture, which is a common theme in educational settings. The punchline "because he wanted to complain about his boring lecture!" makes it clear that the student is not just complaining about the lecture itself but also about the way they were taught and the overall experience of attending class.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03f3-9f5a-654c-8003-22743fc72f17'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-14T13:21:42.702402+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03f3-9080-6c95-8002-8ff8b535b558'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'class', 'joke': "Why did th

In [ ]:
# 'checkpoint_id': '1f1b03f2-2c3e-6dba-8000-1546da8c04c0'

In [68]:
# to resume the workflow where it was left

res = workflow.invoke(None,config=config1)

In [71]:
# Time Travel

workflow.get_state({'configurable':{'thread_id':'1','checkpoint_id': '1f1b03f2-2c3e-6dba-8000-1546da8c04c0'}})


StateSnapshot(values={'topic': 'class'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b03f2-2c3e-6dba-8000-1546da8c04c0'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-14T13:21:03.788984+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03f2-2c3c-68ef-bfff-e50e51fd2947'}}, tasks=(PregelTask(id='ccacf32a-3835-9d45-05d6-894accb8d354', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Why did the class break up?\n\nBecause they couldn't agree on what subject to study!"}),), interrupts=())

In [72]:
# It will start this workflow from the given checkpoint of checkpoint_id 
res = workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id': '1f1b03f2-2c3e-6dba-8000-1546da8c04c0'}})
print(res)

{'topic': 'class', 'joke': "Why did the student go to the teacher's office? Because he wanted to complain about his boring lecture!", 'explaination': 'The joke plays on the idea that the student is complaining about the boring lecture, which is a common theme in educational settings. The punchline "because he wanted to complain about his boring lecture!" makes it clear that the student is not just complaining about the lecture itself but also about the way they were taught and the overall experience of attending class.'}


In [82]:
# update state
workflow.update_state({'configurable':{'thread_id':'1','checkpoint_id': '1f1b03fb-1951-6997-8001-e84d24bd6d17','checkpoint_ns':""}},{'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b0403-71d9-6cc3-8002-b3d620f69774'}}

In [81]:
list(workflow.get_state_history(config1)) 

[StateSnapshot(values={'topic': 'class', 'joke': "Why did the class break up?\n\nBecause they couldn't agree on what subject to study!", 'explaination': 'The joke is about two classes that were supposed to be studying different subjects, but ended up having a disagreement and decided not to continue. The punchline is that their disagreement was so big that it caused them to break up. This joke plays on the idea of students being unable to agree on what subject to study, which can lead to arguments and ultimately causing someone to leave a group or class.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03fe-191c-6e7e-8003-69b054886ba2'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-14T13:26:23.905328+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b03fe-07dc-682b-8002-8eb4c207a017'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'class', 'joke

In [83]:
res = workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id': '1f1b0403-71d9-6cc3-8002-b3d620f69774'}})

print(res)


{'topic': 'samosa', 'joke': 'Why did Samoas go to therapy?\n\nBecause they had too many samosas!', 'explaination': 'The joke is about Samoas (a fictional character from a popular TV show), who is described as having "too many samosas" or "too much of something. " The punchline, "because they had too many samosas!" suggests that Samoas is feeling overwhelmed by the amount of food they are consuming and needs some help to manage their excess.\nIn this joke, the author uses a pun between "samosas" (a type of fried pastry) and "samosa" (a type of spicy Indian curry), which adds humor to the statement. The punchline is that Samoas is feeling overwhelmed by the amount of food they are consuming, so they need some help to manage their excess.\nOverall, the joke is a play on words between Samoas\' name and the food they eat, making it both funny and clever.'}
